# Notebook 03: Tier-2 Training (CatBoost)

## Objective
Classify anomalous traffic into known attack categories:
- **DoS** 
- **BruteForce** 
- **PortScan**

### Model: CatBoost

### Unknown Detection
```
probabilities = model.predict_proba(X)
max_probability = max(probabilities)

if max_probability < 0.65:
    attack = "Unknown"
```

### Zero-Day Testing
Expected behavior:
- DDoS -> Tier-1: Anomaly -> Tier-2: Low confidence -> Output: Unknown
- Botnet -> Tier-1: Anomaly -> Tier-2: Low confidence -> Output: Unknown

### Metrics: Precision, Recall, F1-score, Confusion Matrix

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import joblib
import logging
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_score, recall_score, f1_score
)
from catboost import CatBoostClassifier, Pool
import shap
import matplotlib.pyplot as plt

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

with open('config/config.json', 'r') as f:
    config = json.load(f)

PROCESSED_DIR = os.path.join(config['dataset']['data_dir'], 'processed')
MODELS_DIR = 'models'
os.makedirs(MODELS_DIR, exist_ok=True)

TIER2_CLASSES = config['tier2']['classes']
UNKNOWN_THRESHOLD = config['tier2']['unknown_threshold']

## Step 1: Load Processed Data

In [ ]:
X_train = pd.read_csv(os.path.join(PROCESSED_DIR, 'tier2_X_train.csv'))
X_test = pd.read_csv(os.path.join(PROCESSED_DIR, 'tier2_X_test.csv'))
y_train = pd.read_csv(os.path.join(PROCESSED_DIR, 'tier2_y_train.csv')).values.ravel()
y_test = pd.read_csv(os.path.join(PROCESSED_DIR, 'tier2_y_test.csv')).values.ravel()
feature_order = joblib.load(os.path.join(PROCESSED_DIR, 'feature_order.joblib'))

# Also load zero-day data for evaluation
X_zeroday = pd.read_csv(os.path.join(PROCESSED_DIR, 'zeroday_X.csv'))
y_zeroday = pd.read_csv(os.path.join(PROCESSED_DIR, 'zeroday_y.csv')).values.ravel()

print(f'Tier-2 Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Zero-day eval: {X_zeroday.shape}')
print(f'\nTraining class distribution:')
print(pd.Series(y_train).value_counts())
print(f'\nTest class distribution:')
print(pd.Series(y_test).value_counts())
print(f'\nZero-day labels:')
print(pd.Series(y_zeroday).value_counts())

## Step 2: Preprocessing

In [ ]:
# Clean infinities and NaN
X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
X_test = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)
X_zeroday = X_zeroday.replace([np.inf, -np.inf], np.nan).fillna(0)

# Ensure feature order
X_train = X_train[feature_order]
X_test = X_test[feature_order]
X_zeroday = X_zeroday[feature_order]

# Encode labels
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

print(f'Label mapping:')
for i, cls in enumerate(label_encoder.classes_):
    print(f'  {cls} -> {i}')

# Standardize
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_zeroday_scaled = scaler.transform(X_zeroday)

# Save preprocessor and label encoder
joblib.dump(scaler, os.path.join(MODELS_DIR, 'tier2_preprocessor.joblib'))
joblib.dump(label_encoder, os.path.join(MODELS_DIR, 'tier2_label_encoder.joblib'))
joblib.dump(feature_order, os.path.join(MODELS_DIR, 'tier2_feature_order.joblib'))
print('Preprocessor and label encoder saved.')

## Step 3: Train CatBoost

In [ ]:
# Compute class weights for imbalanced data
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight('balanced', classes=np.unique(y_train_encoded), y=y_train_encoded)
class_weight_dict = {i: w for i, w in enumerate(class_weights)}
print(f'Class weights: {class_weight_dict}')

catboost_params = {
    'iterations': 1000,
    'learning_rate': 0.05,
    'depth': 8,
    'l2_leaf_reg': 3,
    'class_weights': class_weight_dict,
    'eval_metric': 'TotalF1',
    'random_seed': 42,
    'verbose': 100,
    'early_stopping_rounds': 50
}

cat_model = CatBoostClassifier(**catboost_params)

train_pool = Pool(X_train_scaled, y_train_encoded)
val_pool = Pool(X_test_scaled, y_test_encoded)

cat_model.fit(train_pool, eval_set=val_pool)

print('\nCatBoost trained.')

## Step 4: Evaluate on Test Set

In [ ]:
y_pred = cat_model.predict(X_test_scaled)
y_pred_proba = cat_model.predict_proba(X_test_scaled)
y_pred_labels = label_encoder.inverse_transform(y_pred.astype(int))
y_test_labels = label_encoder.inverse_transform(y_test_encoded)

print('CatBoost Classification Report:')
print(classification_report(y_test_labels, y_pred_labels))

cm = confusion_matrix(y_test_labels, y_pred_labels, labels=label_encoder.classes_)
print('Confusion Matrix:')
print(pd.DataFrame(cm, index=label_encoder.classes_, columns=label_encoder.classes_))

precision = precision_score(y_test_labels, y_pred_labels, average='weighted')
recall = recall_score(y_test_labels, y_pred_labels, average='weighted')
f1 = f1_score(y_test_labels, y_pred_labels, average='weighted')
print(f'\nWeighted Precision: {precision:.4f}')
print(f'Weighted Recall: {recall:.4f}')
print(f'Weighted F1: {f1:.4f}')

## Step 5: Evaluate Unknown Detection Threshold

In [ ]:
# Check confidence distribution on test set
max_probas = np.max(y_pred_proba, axis=1)

print('Confidence Distribution on Known Test Set:')
print(f'  Mean: {max_probas.mean():.4f}')
print(f'  Std:  {max_probas.std():.4f}')
print(f'  Min:  {max_probas.min():.4f}')
print(f'  25%:  {np.percentile(max_probas, 25):.4f}')
print(f'  50%:  {np.percentile(max_probas, 50):.4f}')
print(f'  75%:  {np.percentile(max_probas, 75):.4f}')

# Count how many known attacks would be marked Unknown
unknown_count = (max_probas < UNKNOWN_THRESHOLD).sum()
total = len(max_probas)
print(f'\nKnown attacks below threshold {UNKNOWN_THRESHOLD}: {unknown_count}/{total} ({unknown_count/total*100:.1f}%)')

# Test on zero-day data
zeroday_proba = cat_model.predict_proba(X_zeroday_scaled)
zeroday_max_proba = np.max(zeroday_proba, axis=1)
zeroday_pred = cat_model.predict(X_zeroday_scaled)

print(f'\nZero-Day Evaluation:')
print(f'  Mean confidence: {zeroday_max_proba.mean():.4f}')
print(f'  Min confidence: {zeroday_max_proba.min():.4f}')
print(f'  Max confidence: {zeroday_max_proba.max():.4f}')

zeroday_unknown = (zeroday_max_proba < UNKNOWN_THRESHOLD).sum()
zeroday_total = len(zeroday_max_proba)
print(f'  Unknown detections: {zeroday_unknown}/{zeroday_total} ({zeroday_unknown/zeroday_total*100:.1f}%)')
print(f'  Expected: ~100% should be Unknown')

# Breakdown by zero-day type
for label in np.unique(y_zeroday):
    mask = y_zeroday == label
    if mask.sum() > 0:
        probs = zeroday_max_proba[mask]
        unknown_pct = (probs < UNKNOWN_THRESHOLD).sum() / len(probs) * 100
        print(f'  {label}: {unknown_pct:.1f}% classified as Unknown (mean conf: {probs.mean():.4f})')

## Step 6: End-to-End Pipeline Test

Simulate the full Tier-1 -> Tier-2 -> Unknown detection flow on zero-day data.

In [ ]:
# Load Tier-1 models for end-to-end test
lgbm_model = joblib.load(os.path.join(MODELS_DIR, 'tier1_lgbm.joblib'))
iforest = joblib.load(os.path.join(MODELS_DIR, 'tier1_iforest.joblib'))
tier1_scaler = joblib.load(os.path.join(MODELS_DIR, 'tier1_preprocessor.joblib'))
tier1_thresholds = joblib.load(os.path.join(MODELS_DIR, 'tier1_thresholds.joblib'))

LGBM_THRESHOLD = tier1_thresholds['lgbm_threshold']

# Tier-1 on zero-day data
X_zeroday_t1 = X_zeroday[feature_order]
X_zeroday_t1_scaled = tier1_scaler.transform(X_zeroday_t1)

lgbm_proba = lgbm_model.predict_proba(X_zeroday_t1)[:, 1]
if_pred = iforest.predict(X_zeroday_t1_scaled)
if_anomaly = if_pred == -1
lgbm_anomaly = lgbm_proba > LGBM_THRESHOLD
tier1_anomaly = lgbm_anomaly | if_anomaly

print(f'Tier-1 on Zero-Day Data:')
print(f'  Total samples: {len(X_zeroday)}')
print(f'  Detected as anomaly: {tier1_anomaly.sum()} ({tier1_anomaly.mean()*100:.1f}%)')
print(f'  Expected: Should detect most as anomaly (high recall)')

# Tier-2 on samples that passed Tier-1
if tier1_anomaly.sum() > 0:
    X_zeroday_t2 = X_zeroday[tier1_anomaly]
    X_zeroday_t2_scaled = scaler.transform(X_zeroday_t2)

    t2_proba = cat_model.predict_proba(X_zeroday_t2_scaled)
    t2_max_proba = np.max(t2_proba, axis=1)
    t2_predictions = cat_model.predict(X_zeroday_t2_scaled)

    unknown_mask = t2_max_proba < UNKNOWN_THRESHOLD
    print(f'\nTier-2 on Detected Anomalies:')
    print(f'  Unknown: {unknown_mask.sum()} ({unknown_mask.mean()*100:.1f}%)')
    print(f'  Classified: {(~unknown_mask).sum()} ({(~unknown_mask).mean()*100:.1f}%)')

    if (~unknown_mask).sum() > 0:
        classified = t2_predictions[~unknown_mask]
        classified_labels = label_encoder.inverse_transform(classified.astype(int))
        print(f'  Misclassified as: {pd.Series(classified_labels).value_counts().to_dict()}')

print('\n--- Full Pipeline Summary ---')
print(f'Zero-day samples: {len(X_zeroday)}')
print(f'Tier-1 detected: {tier1_anomaly.sum()} anomalies')
print(f'Tier-2 classified as Unknown: {unknown_mask.sum() if tier1_anomaly.sum() > 0 else 0}')
print(f'Correctly identified as unknown: {(unknown_mask.sum() / tier1_anomaly.sum() * 100) if tier1_anomaly.sum() > 0 else 0:.1f}%')

## Step 7: SHAP Explanation

In [ ]:
cat_explainer = shap.TreeExplainer(cat_model)
cat_shap_values = cat_explainer.shap_values(X_test_scaled[:200])

shap.summary_plot(cat_shap_values, X_test[:200], plot_type='bar')
plt.title('Tier-2 CatBoost Feature Importance (SHAP)')
plt.tight_layout()
plt.savefig('models/tier2_shap_summary.png', dpi=150)
plt.show()

joblib.dump(cat_explainer, os.path.join(MODELS_DIR, 'tier2_shap_explainer.joblib'))
print('SHAP explainer saved.')

## Step 8: Save Models & Artifacts

In [ ]:
joblib.dump(cat_model, os.path.join(MODELS_DIR, 'tier2_catboost.joblib'))
joblib.dump(scaler, os.path.join(MODELS_DIR, 'tier2_preprocessor.joblib'))
joblib.dump(label_encoder, os.path.join(MODELS_DIR, 'tier2_label_encoder.joblib'))
joblib.dump(feature_order, os.path.join(MODELS_DIR, 'tier2_feature_order.joblib'))

thresholds = {
    'unknown_threshold': UNKNOWN_THRESHOLD,
    'classes': list(label_encoder.classes_)
}
joblib.dump(thresholds, os.path.join(MODELS_DIR, 'tier2_thresholds.joblib'))

# Update config
config['tier2']['classes'] = list(label_encoder.classes_)
with open('config/config.json', 'w') as f:
    json.dump(config, f, indent=4)

print('Tier-2 models and artifacts saved:')
print(f'  tier2_catboost.joblib')
print(f'  tier2_preprocessor.joblib')
print(f'  tier2_label_encoder.joblib')
print(f'  tier2_feature_order.joblib')
print(f'  tier2_thresholds.joblib')
print(f'  tier2_shap_explainer.joblib')
print(f'  Unknown threshold: {UNKNOWN_THRESHOLD}')
print(f'  Classes: {list(label_encoder.classes_)}')